# Vaidya SLM: QLoRA Fine-tuning
This notebook covers the fine-tuning of `TinyLlama/TinyLlama-1.1B-Chat-v1.0` (as a lightweight <3B alternative) using PEFT/LoRA specifically on Ayurvedic Instruction datasets.

In [ ]:
!pip install -q -U peft transformers datasets trl accelerate bitsandbytes

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# 1. Load Model & Tokenizer
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    load_in_4bit=True,
    device_map="auto"
)
model = prepare_model_for_kbit_training(model)

# 2. LoRA Config for PEFT
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

# 3. Load processed dataset
try:
    dataset = load_dataset("json", data_files="../../data/processed/ayurvedic_dataset.json", split="train")
except:
    # Fallback to sample data for Colab
    dataset = load_dataset("json", data_files="ayurvedic_dataset.json", split="train")

# 4. Format Prompt
def format_instruction(sample):
    # Formats inputs according to Chat Template or Instruct formats
    instruction = sample['instruction']
    in_text = sample['input']
    out_text = f"Dosha: {sample['output']['dosha']}\nReason: {sample['output']['reason']}\nRemedy: {sample['output']['remedy']}"
    return f"### Instruction:\n{instruction}\n\n### Input:\n{in_text}\n\n### Response:\n{out_text}"

# 5. Setup Trainer (with metrics, validation split if available, checkpoints)
training_args = TrainingArguments(
    output_dir="./vaidya-slm-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    save_steps=50,
    logging_steps=10,
    learning_rate=2e-4,
    fp16=True,
    max_steps=200, # Extend up to full epochs based on dataset size
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    formatting_func=format_instruction,
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args
)

# 6. Train and Save
trainer.train()
trainer.model.save_pretrained("./vaidya-slm-final")
tokenizer.save_pretrained("./vaidya-slm-final")
print("Training Complete. LoRA weights saved to ./vaidya-slm-final")
